# NetraTrust - train the DR backbone (Kaggle T4)

`convnext_tiny.in12k_ft_in1k` @ 512px, 5 classes, on APTOS.

**Read before running.** This notebook is the ONLY place the DR backbone is
trained. The build box has no GPU, so `netra/models/train.py` is only ever
CPU-smoked there (two steps, to prove the loop runs). Everything real happens
here.

Rules this notebook must not break - they are in `.claude/rules/ml.md`:

1. **Splits come from `data/splits/*.json`.** They are frozen and committed.
   Never regenerate them, never re-shuffle, and never call `train_test_split`.
2. **The test split is touched exactly once, at the end.** Early stopping and
   model selection use the **calibration** split. `collect()` refuses
   `split="test"` unless you pass `allow_test=True`, which nothing below does.
3. **Every reported number comes from `netra/eval/metrics.py`.** Do not compute
   a kappa inline in a cell and quote it.
4. **Preprocessing is background removal + resize only.** No Ben Graham, no
   CLAHE. A controlled ablation put both BELOW plain crop+resize (0.8260 and
   0.8238 vs 0.8310); resolution is what pays.
5. **Report the retained fraction at every filtering step.** `collect()` returns
   it; print it. "Reasonable" preprocessing has been documented to silently
   discard 80-94% of public corpora.

Expected QWK on a local APTOS split: **0.88-0.92**. Anything **above 0.95 means
a leaked split** - stop and find the leak rather than celebrating.

## 1. Attach the data and the repo

Add the APTOS dataset via *Add Data*, point `IMAGES_DIR` at the images, and make
sure this repo's `data/splits/` and `data/manifest.csv` are present. The splits
are the committed ones; they are not regenerated here.

In [ ]:
import json
import pathlib
import sys

import torch

REPO = pathlib.Path("/kaggle/working/netratrust")   # clone or copy the repo here
IMAGES_DIR = pathlib.Path("/kaggle/input/aptos2019/train_images/train_images")
OUT = pathlib.Path("/kaggle/working/checkpoints")
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO))
assert (REPO / "data" / "splits" / "train.json").exists(), (
    "the frozen splits must be present; do NOT regenerate them here"
)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 2. Confirm the splits, and that nothing leaks

This re-checks the invariant rather than trusting it. Note the honest limit:
APTOS publishes no patient id, so `patient_id == image_id` and this check is
weaker than it looks for this corpus. That is recorded in
`scripts/build_manifest.py` and in `memory.md`.

In [ ]:
from netra.models.dataset import read_split

SPLITS = REPO / "data" / "splits"
MANIFEST = REPO / "data" / "manifest.csv"

splits = {name: set(read_split(name, SPLITS)) for name in ("train", "calib", "test")}
for a in splits:
    for b in splits:
        if a < b:
            overlap = splits[a] & splits[b]
            assert not overlap, f"LEAK between {a} and {b}: {sorted(overlap)[:5]}"
print({k: len(v) for k, v in splits.items()})
print("no image appears in two folds")

## 3. Retained fractions, before training anything

If `retained_fraction` is not close to 1.0 the experiment is smaller than it
looks. Find out why before training, not after.

In [ ]:
from netra.models.dataset import collect

train_examples, train_report = collect(
    "train", IMAGES_DIR, splits_dir=SPLITS, manifest=MANIFEST
)
calib_examples, calib_report = collect(
    "calib", IMAGES_DIR, splits_dir=SPLITS, manifest=MANIFEST
)
print(json.dumps({"train": train_report, "calib": calib_report}, indent=2))
assert train_report["retained_fraction"] > 0.95, "too many training images missing"
# NOTE: the test split is deliberately NOT collected here.

## 4. Train

Model selection is on the **calibration** fold. A checkpoint is written only
when calibration QWK improves, and it carries `trained: true` - which is what
`DRClassifier` reads to decide whether the weights may be reported as a model
at all. A smoke artifact carries `trained: false` and reports itself as
`:untrained` forever.

In [ ]:
from netra.models.train import train

result = train(
    images_dir=IMAGES_DIR,
    epochs=12,
    batch_size=16,          # 512px on a T4; drop to 8 if it OOMs
    lr=3e-4,
    device="cuda",
    smoke=False,
    out=OUT / "convnext_tiny_aptos.pt",
    pretrained=True,
)
print(json.dumps({k: v for k, v in result.items() if k != "history"}, indent=2))

## 5. QWK per epoch

Plotted from the history the trainer returned. The red line is the
leaked-split alarm, not a target.

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in result["history"]]
qwk = [h.get("calib", {}).get("qwk") for h in result["history"]]

plt.figure(figsize=(6, 3.5))
plt.plot(epochs, qwk, marker="o")
plt.axhline(0.95, ls="--", lw=1, color="crimson")
plt.annotate(
    "above this: suspect a leaked split",
    (epochs[0], 0.955),
    color="crimson",
    fontsize=8,
)
plt.xlabel("epoch")
plt.ylabel("calibration QWK")
plt.ylim(0, 1)
plt.title("Model selection on the CALIBRATION fold")
plt.tight_layout()
plt.show()

## 6. Conformal calibration

Fit the abstention wrapper on the **calibration** fold, never on test. The
abstention rule is not "the set is large" - it is "the set spans more than one
CLINICAL ACTION". `{0,1}` and `{0,3}` are both size 2 and only the second is
alarming.

In [ ]:
import numpy as np

from netra.models.abstain import AbstentionWrapper, PrefitProbaEstimator
from netra.models.dataset import load_batch
from netra.models.dr import DRClassifier

clf = DRClassifier(checkpoint=OUT / "convnext_tiny_aptos.pt", device="cuda")
assert clf.trained, "refusing to conformalise weights that are not marked trained"

images, labels = load_batch(calib_examples)
proba = np.stack([clf.predict_proba(img) for img in images])

wrapper = AbstentionWrapper(PrefitProbaEstimator(n_classes=5))
wrapper.conformalize(proba, labels)
print("fitted on", len(labels), "calibration cases at", wrapper.confidence_level)

report = wrapper.coverage_report(proba, labels)
print(json.dumps(report, indent=2))

## 7. Download the checkpoint, then fit thresholds

Save `convnext_tiny_aptos.pt`, put it where `NETRA_DR_CHECKPOINT` points, and
run `python -m netra.models.fit_thresholds` so `config/thresholds.yaml` carries
values fitted on the calibration fold rather than the provisional placeholders
this repo ships.

**The test split stays untouched until the very end** - and then once.

In [ ]:
checkpoint = OUT / "convnext_tiny_aptos.pt"
print("checkpoint:", round(checkpoint.stat().st_size / 1e6, 1), "MB")
print("next, on the build box:")
print("  NETRA_DR_CHECKPOINT=<path> python -m netra.models.fit_thresholds --images <dir>")